# 01 — 학습 데이터 생성

**목적:** Adzuna에서 채용공고를 수집하고, Claude Haiku로 기술스택을 레이블링해 `train.jsonl` / `test.jsonl`을 만든다.

**런타임:** CPU 가능 (GPU 불필요)  
**예상 비용:** Haiku 기준 650 공고 약 $0.25  
**출력 위치:** Colab이면 Google Drive, 로컬이면 `finetune/dataset/`

---
### 선행 조건 (3개 모두 필수)
Colab 왼쪽 사이드바 🔑 **Secrets** 에 등록하거나, 로컬 `.env`에 설정하세요.
```
ANTHROPIC_API_KEY   # Claude Haiku 레이블 생성
ADZUNA_APP_ID       # Adzuna API (developer.adzuna.com 무료 발급)
ADZUNA_APP_KEY
```

In [1]:
# 의존성 설치 (Colab 전용 — 로컬은 requirements.txt 사용)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q openai httpx python-dotenv

In [ ]:
import json
import os
import time
import random
from pathlib import Path

import httpx

# ── 환경 설정 ──────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY', '')
    ADZUNA_APP_ID  = userdata.get('ADZUNA_APP_ID', '')
    ADZUNA_APP_KEY = userdata.get('ADZUNA_APP_KEY', '')
    OUTPUT_DIR = Path('/content/drive/MyDrive/job-skill-ft/dataset')
else:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
    ADZUNA_APP_ID  = os.getenv('ADZUNA_APP_ID', '')
    ADZUNA_APP_KEY = os.getenv('ADZUNA_APP_KEY', '')
    # VS Code는 노트북 파일이 있는 디렉토리(finetune/)를 CWD로 설정
    OUTPUT_DIR = Path('dataset')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [k for k, v in {
    'OPENAI_API_KEY': OPENAI_API_KEY,
    'ADZUNA_APP_ID':  ADZUNA_APP_ID,
    'ADZUNA_APP_KEY': ADZUNA_APP_KEY,
}.items() if not v]

if missing:
    raise EnvironmentError(f'필수 환경변수 누락: {missing}\n.env 파일 또는 Colab Secrets를 확인하세요.')

print('✅ 환경변수 확인 완료')
print(f'저장 경로: {OUTPUT_DIR.resolve()}')

In [3]:
# ── Adzuna 공고 수집 함수 ──────────────────────────────────────
SEARCH_QUERIES = [
    'ai engineer llm',
    'machine learning engineer',
    'llm engineer langchain',
    'nlp engineer transformer',
    'ml engineer python',
    'backend engineer python fastapi',
]

def fetch_adzuna_page(query: str, page: int = 1, n: int = 50) -> list[dict]:
    url = f'https://api.adzuna.com/v1/api/jobs/gb/search/{page}'
    params = {
        'app_id': ADZUNA_APP_ID,
        'app_key': ADZUNA_APP_KEY,
        'what': query,
        'results_per_page': n,
        'content-type': 'application/json',
    }
    resp = httpx.get(url, params=params, timeout=20)
    resp.raise_for_status()
    return resp.json().get('results', [])

In [4]:
# ── 공고 수집 실행 ─────────────────────────────────────────────
TARGET_COUNT = 700  # 최대 수집 목표

all_jobs: dict[str, dict] = {}  # id → job, 중복 제거

print('Adzuna API로 수집 중...')
for query in SEARCH_QUERIES:
    for page in range(1, 5):  # 페이지당 50개 × 4 = 200개/쿼리
        try:
            jobs = fetch_adzuna_page(query, page=page, n=50)
            if not jobs:
                break
            for j in jobs:
                if j.get('id') and j.get('description'):
                    all_jobs[j['id']] = j
            print(f'  query={query!r:35} page={page} → 누적 {len(all_jobs)}개')
            time.sleep(0.5)  # API rate limit 존중
        except Exception as e:
            print(f'  [skip] {query} page={page}: {e}')
            break
        if len(all_jobs) >= TARGET_COUNT:
            break
    if len(all_jobs) >= TARGET_COUNT:
        break

jobs_list = list(all_jobs.values())
jobs_list = [j for j in jobs_list if len(j.get('description', '')) >= 100]

random.seed(42)
random.shuffle(jobs_list)

print(f'\n수집 완료: {len(jobs_list)}개 (중복·내용 부족 제거 후)')

Adzuna API로 수집 중...
  query='ai engineer llm'                   page=1 → 누적 50개
  query='ai engineer llm'                   page=2 → 누적 78개
  query='machine learning engineer'         page=1 → 누적 126개
  query='machine learning engineer'         page=2 → 누적 176개
  query='machine learning engineer'         page=3 → 누적 226개
  query='machine learning engineer'         page=4 → 누적 276개
  query='llm engineer langchain'            page=1 → 누적 278개
  query='ml engineer python'                page=1 → 누적 299개
  query='backend engineer python fastapi'   page=1 → 누적 323개

수집 완료: 323개 (중복·내용 부족 제거 후)


In [5]:
# ── OpenAI 레이블 생성 함수 ────────────────────────────────────
# skill_extractor.py의 extract_skills() 프롬프트와 동일한 형식 사용

INSTRUCTION = (
    'Extract technical skills from the job posting below. '
    'Return valid JSON only with this exact schema — no markdown fences, no commentary:\n'
    '{"job_title": "string", "skills": [{"raw": "string", '
    '"category": "language|framework|database|cloud|tool|concept", '
    '"importance": "required|preferred"}]}'
)


def label_job(job: dict, client) -> dict | None:
    """GPT-4o-mini로 공고 레이블 생성. 실패 시 None 반환."""
    title = job.get('title', '')
    description = job.get('description', '')[:2000]
    user_content = f'Title: {title}\n\nDescription:\n{description}'

    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': f'{INSTRUCTION}\n\n{user_content}'
            }],
        )
        raw = resp.choices[0].message.content.strip()
        raw = raw.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(raw)

        if not parsed.get('skills'):
            return None  # 기술이 1개 이상이어야 통과

        return {
            'instruction': INSTRUCTION,
            'input': user_content,
            'output': json.dumps(parsed, ensure_ascii=False, separators=(',', ':')),
        }
    except Exception as e:
        print(f'  [skip] {title[:40]}: {e}')
        return None


print('레이블 생성 함수 정의 완료 (GPT-4o-mini)')

레이블 생성 함수 정의 완료 (GPT-4o-mini)


In [6]:
# ── 레이블 생성 실행 ───────────────────────────────────────────
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
samples: list[dict] = []

print(f'GPT-4o-mini로 {len(jobs_list)}개 레이블 생성 시작...')
print(f'예상 비용: 약 ${len(jobs_list) * 0.00015:.2f}')  # gpt-4o-mini 기준

for i, job in enumerate(jobs_list):
    result = label_job(job, client)
    if result:
        samples.append(result)
    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(jobs_list)}] 완료 — 유효 샘플: {len(samples)}개')
    time.sleep(0.05)

print(f'\n최종 유효 샘플: {len(samples)}개 (제거: {len(jobs_list) - len(samples)}개)')

GPT-4o-mini로 323개 레이블 생성 시작...
예상 비용: 약 $0.05
  [skip] Machine Learning Engineer: Expecting ',' delimiter: line 1 column 572 (char 571)
  [50/323] 완료 — 유효 샘플: 42개
  [100/323] 완료 — 유효 샘플: 81개
  [150/323] 완료 — 유효 샘플: 123개
  [200/323] 완료 — 유효 샘플: 168개
  [skip] Senior Machine Learning Engineer, Recomm: Expecting ',' delimiter: line 1 column 588 (char 587)
  [250/323] 완료 — 유효 샘플: 211개
  [300/323] 완료 — 유효 샘플: 252개

최종 유효 샘플: 271개 (제거: 52개)


In [ ]:
# ── 데이터 품질 후처리 ─────────────────────────────────────────────
# 1. 추상 직무 도메인 개념 제거 (Machine Learning, AI, Data Analysis 등)
# 2. 기술명 정규화 (machine learning → Python, AWS 등 표준 표기 통일)
# 3. 정규화 후 대소문자 중복 제거

ABSTRACT_CONCEPTS = {
    "machine learning", "ai", "artificial intelligence", "deep learning",
    "data science", "data analysis", "data engineering", "computer science",
    "software engineering", "software development", "cloud computing",
    "distributed systems", "big data", "analytics", "business intelligence",
    "predictive modeling", "statistical analysis", "data modeling",
    "data visualization", "optimization", "scalability", "system design",
    "microservices", "agile", "ci/cd", "devops", "algorithm", "algorithms",
    "natural language processing", "computer vision", "reinforcement learning",
    "data collection", "data processing", "model development", "api development",
    "backend development", "web development", "mobile development",
    "problem solving", "communication", "teamwork", "collaboration",
    "ml", "software applications", "enterprise software", "technical skills",
    "quantitative finance", "ai systems", "cloud services", "data pipelines",
    "data infrastructure",
}

_ALIASES: dict[str, str] = {
    "react.js": "React", "reactjs": "React",
    "node.js": "Node.js", "nodejs": "Node.js",
    "fastapi": "FastAPI",
    "langchain": "LangChain", "lang chain": "LangChain",
    "langgraph": "LangGraph",
    "python": "Python", "python3": "Python",
    "tensorflow": "TensorFlow", "tf": "TensorFlow",
    "pytorch": "PyTorch", "torch": "PyTorch",
    "aws": "AWS", "amazon web services": "AWS",
    "gcp": "GCP", "google cloud platform": "GCP", "google cloud": "GCP",
    "sql": "SQL", "llm": "LLM", "nlp": "NLP", "rag": "RAG", "rlhf": "RLHF",
    "docker": "Docker",
    "kubernetes": "Kubernetes", "k8s": "Kubernetes",
    "postgresql": "PostgreSQL", "postgres": "PostgreSQL",
    "mongodb": "MongoDB", "neo4j": "Neo4j",
    "elasticsearch": "Elasticsearch",
    "hugging face": "Hugging Face", "huggingface": "Hugging Face",
    "openai": "OpenAI", "anthropic": "Anthropic",
    "mlops": "MLOps",
    "scikit-learn": "scikit-learn", "sklearn": "scikit-learn",
    "pandas": "pandas", "numpy": "NumPy",
    "github": "GitHub", "git": "Git",
    "rest api": "REST API", "restful api": "REST API", "restful": "REST API",
    "graphql": "GraphQL",
    "typescript": "TypeScript", "javascript": "JavaScript",
    "html": "HTML", "css": "CSS",
    "chroma": "Chroma", "chromadb": "Chroma",
    "weaviate": "Weaviate", "pinecone": "Pinecone",
    "langfuse": "Langfuse", "ragas": "RAGAS",
    "lora": "LoRA", "qlora": "QLoRA",
    "transformer": "Transformer", "transformers": "Transformers",
    "redis": "Redis", "flask": "Flask", "django": "Django",
    "spark": "Spark", "apache spark": "Spark",
    "kafka": "Kafka", "apache kafka": "Kafka",
    "airflow": "Airflow", "azure": "Azure", "microsoft azure": "Azure",
    "gpt": "GPT", "bert": "BERT", "ci/cd": "CI/CD",
    "peft": "PEFT", "vllm": "vLLM", "ollama": "Ollama",
    "haystack": "Haystack",
    "llamaindex": "LlamaIndex", "llama index": "LlamaIndex",
    "xgboost": "XGBoost", "lightgbm": "LightGBM",
    "matplotlib": "Matplotlib", "seaborn": "Seaborn",
    "jupyter": "Jupyter", "linux": "Linux", "bash": "Bash",
}


def _smart_title(s: str) -> str:
    """전부 대문자인 단어(약어)는 그대로 유지, 나머지만 capitalize."""
    words = s.strip().split()
    result = []
    for w in words:
        if w.isupper() and len(w) >= 2:
            result.append(w)                      # AWS, LLM, NLP 등
        elif w and w[0].isupper() and not w[1:].islower():
            result.append(w)                      # PyTorch, TensorFlow 등 이미 표준 표기
        else:
            result.append(w.capitalize())
    return " ".join(result)


def _normalize_raw(raw: str) -> str:
    lower = raw.strip().lower()
    if lower in _ALIASES:
        return _ALIASES[lower]
    return _smart_title(raw)


def _is_abstract_concept(raw: str, category: str) -> bool:
    return category == "concept" and raw.strip().lower() in ABSTRACT_CONCEPTS


def postprocess_samples(raw_samples: list[dict]) -> list[dict]:
    """추상 개념 제거 + 기술명 정규화 + 중복 제거."""
    processed = []
    for s in raw_samples:
        try:
            parsed = json.loads(s["output"])
            skills = parsed.get("skills", [])

            # 1. 추상 직무 도메인 개념 제거
            skills = [sk for sk in skills
                      if not _is_abstract_concept(sk.get("raw", ""), sk.get("category", ""))]

            # 2. raw 필드 정규화
            for sk in skills:
                sk["raw"] = _normalize_raw(sk["raw"])

            # 3. 정규화 후 중복 제거 (lower-case 기준 첫 번째 유지)
            seen: set[str] = set()
            deduped = []
            for sk in skills:
                key = sk["raw"].lower()
                if key not in seen:
                    seen.add(key)
                    deduped.append(sk)

            if not deduped:
                continue  # 기술 0개면 샘플 제거

            parsed["skills"] = deduped
            processed.append({
                "instruction": s["instruction"],
                "input": s["input"],
                "output": json.dumps(parsed, ensure_ascii=False, separators=(",", ":")),
            })
        except Exception:
            continue
    return processed


samples = postprocess_samples(samples)
print(f'후처리 완료: {len(samples)}개 샘플')

In [7]:
# ── 학습/테스트 분할 + 저장 ────────────────────────────────────
random.seed(42)
random.shuffle(samples)

TEST_SIZE = min(100, max(20, len(samples) // 7))  # 약 13%
test_samples  = samples[:TEST_SIZE]
train_samples = samples[TEST_SIZE:]

def save_jsonl(data: list[dict], path: Path) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'  저장: {path} ({len(data)}개)')

save_jsonl(train_samples, OUTPUT_DIR / 'train.jsonl')
save_jsonl(test_samples,  OUTPUT_DIR / 'test.jsonl')

print(f'\n완료!  Train {len(train_samples)}개 / Test {len(test_samples)}개')

  저장: finetune/dataset/train.jsonl (233개)
  저장: finetune/dataset/test.jsonl (38개)

완료!  Train 233개 / Test 38개


In [8]:
# ── 통계 확인 ─────────────────────────────────────────────────
from collections import Counter

all_skills = []
all_categories = []
skill_counts = []

for s in samples:
    try:
        parsed = json.loads(s['output'])
        skill_list = parsed.get('skills', [])
        skill_counts.append(len(skill_list))
        for sk in skill_list:
            all_skills.append(sk.get('raw', ''))
            all_categories.append(sk.get('category', ''))
    except Exception:
        pass

print('=== 데이터셋 통계 ===')
print(f'총 샘플       : {len(samples)}')
print(f'평균 기술 수  : {sum(skill_counts)/len(skill_counts):.1f} 개/공고')
print(f'총 기술 추출  : {len(all_skills)} 개')
print()
print('카테고리 분포:')
for cat, cnt in Counter(all_categories).most_common():
    print(f'  {cat:12} {cnt:4}  ({cnt/len(all_categories)*100:.1f}%)')
print()
print('빈출 기술 Top 15:')
for skill, cnt in Counter(all_skills).most_common(15):
    print(f'  {skill:20} {cnt}')

# 샘플 1개 미리 보기
print('\n=== 샘플 예시 (train[0]) ===')
ex = train_samples[0]
print('INPUT :', ex['input'][:200], '...')
print('OUTPUT:', ex['output'][:300])

=== 데이터셋 통계 ===
총 샘플       : 271
평균 기술 수  : 5.2 개/공고
총 기술 추출  : 1408 개

카테고리 분포:
  concept       771  (54.8%)
  framework     208  (14.8%)
  language      176  (12.5%)
  tool          101  (7.2%)
  cloud          98  (7.0%)
  database       54  (3.8%)

빈출 기술 Top 15:
  Python               153
  Machine Learning     106
  TensorFlow           93
  PyTorch              72
  machine learning     69
  AI                   53
  SQL                  48
  AWS                  47
  Data Analysis        33
  FastAPI              23
  Docker               17
  data analysis        16
  Cloud Computing      16
  cloud computing      13
  LLM                  12

=== 샘플 예시 (train[0]) ===
INPUT : Title: SWE - Machine Learning Engineer, Photos

Description:
Do you have a passion for building the best photo app experience in the world? iPhone is the most popular camera in the world! Seamless int ...
OUTPUT: {"job_title":"SWE - Machine Learning Engineer, Photos","skills":[{"raw":"machine learning","ca